In [115]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from requests import delete

In [116]:
input_file = "Student_Dataset.csv"
output_file = "Cleaned_Student_Dataset.csv"

df = pd.read_csv(
    input_file,
    delimiter=",",
    header=None,
    index_col=0,
    dtype=str,
    names=["ID", "Colors", "Testimony"]
)
first_len = len(df)
print("Len before claening :", first_len)

Len before claening : 1011


#### get sentences

In [117]:
sentences = []
def parse_line(line: str) -> str | None:
    line = line.rstrip("\n")
    if not line:
        return None
    try:
        _id, rest = line.split(",", 1)
    except ValueError:
        return None
    rest = rest.lstrip()
    if rest.startswith("0x000000,"):
        rest = rest[len("0x000000,"):].lstrip()

    rest = rest.strip()
    if len(rest) >= 2 and rest[0] == '"' and rest[-1] == '"':
        rest = rest[1:-1].replace('""', '"')

    return rest or None

sentences = []
with open(input_file, "r", encoding="utf-8") as f:
    for line in f:
        s = parse_line(line)
        if s:
            sentences.append(s)

len_sentences = first_len - len(sentences)
print("len total :", len(sentences))
print("lost :", len_sentences, "Lines")


len total : 1010
lost : 1 Lines


###### drop duplicate

In [118]:
def drop_duplicates(lst):
    return list(dict.fromkeys(lst))

sentences = drop_duplicates(sentences)

after_count = len_sentences - len(sentences)
print(after_count)


-1000


#### delete useless words
example in the sentence : "I have a persistent feeling of thirst, especially after meals. I also lose weight without trying."
we keep : "persistent feeling thirst, especially after meals.  lose weight without trying."

In [119]:
stop_words = {
    'the', 'a', 'an', 'and', 'or', 'but', 'if', 'then', 'else', 'is', 'are',
    'was', 'were', 'be', 'been', 'being', 'to', 'of', 'in', 'for', 'with',
    'on', 'at', 'by', 'from', 'up', 'down', 'this', 'that', 'it', 'its', 'i'
}

cleaned_sentences = []

for s in sentences:
    raw_sentence = s
    words = raw_sentence.lower().split()
    filtered_words = [w for w in words if w not in stop_words]
    clean_sentence = " ".join(filtered_words)
    cleaned_sentences.append(clean_sentence)


Create new dataframe without duplicate rows or missing value

In [120]:
df_final = pd.DataFrame({
    "Color": ["0x000000"] * len(cleaned_sentences),
    "Sentences": cleaned_sentences
})

print("Before :", first_len)
print("After :", len(df_final))
print("Lost : ", first_len - len(df_final), "Lines")

Before : 1011
After : 1001
Lost :  10 Lines


#### Put the dataframe in a csv file

In [121]:
df_final.to_csv(output_file, index=True, index_label="Index")